# 03. Intervals

An interval is `[start, end]`. Problems: **merge**, **insert**, **intersect**, **remove overlaps**, **meetings**. Almost all start with **sorting**.

**Learning order:** REAL LIFE → OVERLAP TEST → MERGE → INSERT → INTERSECTION → REMOVE OVERLAPS → ARROWS → MEETINGS → SORT BY START OR END → INTERVIEW THINKING

## 1. Real life and the overlap test

Your **calendar** is a list of intervals. Two meetings `[a, b]` and `[c, d]` **overlap** if `a < d and c < b` (strict if touching ends do not count, `<=` if they do). They are **disjoint** if `b <= c` or `d <= a`.

**Rule of thumb:** after sorting by **start**, you only need to compare each interval with the **previous (merged)** one.

In [1]:
def merge_intervals(intervals):
    res = []
    for start, end in sorted(intervals):
        if res and start <= res[-1][1]:
            res[-1][1] = max(res[-1][1], end)       # overlap: extend
        else:
            res.append([start, end])
    return res

print(merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]))   # [[1, 6], [8, 10], [15, 18]]

[[1, 6], [8, 10], [15, 18]]


## 2. Insert Interval (list is already sorted)

Three phases in one pass:
1. Add all intervals that end **before** the new one starts.
2. **Merge** all intervals that overlap the new one.
3. Add the rest.

In [2]:
def insert_interval(intervals, new):
    res, i, n = [], 0, len(intervals)
    while i < n and intervals[i][1] < new[0]:
        res.append(intervals[i]); i += 1
    while i < n and intervals[i][0] <= new[1]:
        new = [min(new[0], intervals[i][0]), max(new[1], intervals[i][1])]
        i += 1
    res.append(new)
    res.extend(intervals[i:])
    return res

print(insert_interval([[1, 3], [6, 9]], [2, 5]))                                  # [[1, 5], [6, 9]]
print(insert_interval([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]))       # [[1, 2], [3, 10], [12, 16]]

[[1, 5], [6, 9]]
[[1, 2], [3, 10], [12, 16]]


## 3. Interval List Intersections (two pointers)

Two sorted lists of disjoint intervals. For `A[i]` and `B[j]`: the intersection is `[max(starts), min(ends)]` if valid. Then advance the pointer whose interval **ends first** (it cannot intersect anything else).

In [3]:
def interval_intersection(A, B):
    i = j = 0
    res = []
    while i < len(A) and j < len(B):
        lo, hi = max(A[i][0], B[j][0]), min(A[i][1], B[j][1])
        if lo <= hi:
            res.append([lo, hi])
        if A[i][1] < B[j][1]:
            i += 1
        else:
            j += 1
    return res

print(interval_intersection([[0, 2], [5, 10], [13, 23], [24, 25]], [[1, 5], [8, 12], [15, 24], [25, 26]]))
# [[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]

[[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]


## 4. Non-overlapping Intervals (minimum removals)

Equivalent to: **keep the maximum number of non-overlapping intervals**. Greedy: sort by **end**, always keep the interval that ends earliest (leaves the most room). Removals = `n - kept`.

In [4]:
def erase_overlap_intervals(intervals):
    intervals.sort(key=lambda x: x[1])
    kept, last_end = 0, float("-inf")
    for start, end in intervals:
        if start >= last_end:
            kept += 1
            last_end = end
    return len(intervals) - kept

print(erase_overlap_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]), erase_overlap_intervals([[1, 2], [1, 2], [1, 2]]), erase_overlap_intervals([[1, 2], [2, 3]]))   # 1 2 0

1 2 0


## 5. Minimum Arrows to Burst Balloons

Balloons are intervals on a line. One arrow at `x` bursts every balloon with `start <= x <= end`. Same greedy: sort by **end**, shoot at the end of the first balloon, skip all balloons that start at or before it.

In [5]:
def find_min_arrow_shots(points):
    points.sort(key=lambda p: p[1])
    arrows, arrow_x = 0, float("-inf")
    for start, end in points:
        if start > arrow_x:
            arrows += 1
            arrow_x = end
    return arrows

print(find_min_arrow_shots([[10, 16], [2, 8], [1, 6], [7, 12]]), find_min_arrow_shots([[1, 2], [3, 4], [5, 6], [7, 8]]), find_min_arrow_shots([[1, 2], [2, 3], [3, 4], [4, 5]]))   # 2 4 2

2 4 2


## 6. Meeting Rooms I

Can one person attend all meetings? Sort by start and check that each meeting starts **after the previous one ends**. (Rooms needed = Meeting Rooms II, with a heap or sweep line.)

In [6]:
def can_attend_meetings(intervals):
    intervals.sort()
    return all(intervals[i][0] >= intervals[i - 1][1] for i in range(1, len(intervals)))

print(can_attend_meetings([[0, 30], [5, 10], [15, 20]]), can_attend_meetings([[7, 10], [2, 4]]))   # False True

False True


## 7. Sort by start or by end?

| Goal | Sort by | Why |
|---|---|---|
| merge overlapping / insert / calendar scan | **start** | overlaps become adjacent |
| maximum non-overlapping, minimum removals, minimum arrows, activity selection | **end** | finishing early leaves the most room |
| minimum rooms | start (+ heap of ends) or sweep line | |

## 8. Interview thinking

**Clarify:** do touching intervals `[1,2]` and `[2,3]` overlap? Are they sorted? Can they be empty?

**Say:** "I'll sort by start (or end) in O(n log n), then do one pass comparing with the last interval."

**Common mistakes:** sorting by the wrong key, mutating the input, `<` vs `<=` on touching intervals, forgetting `max` of ends when merging (`[1,10]` then `[2,3]`), not copying lists when merging.

## Quick Revision

- Overlap: `a < d and c < b` (strictness depends on the problem).
- Merge: sort by start, extend with `max(end)`.
- Insert: before / merge / after.
- Intersection: `[max(starts), min(ends)]`, advance the one that ends first.
- Max non-overlapping / arrows: sort by **end**.